In [ ]:
# ================================
# INSTALLS
# ================================

!pip install pymupdf pillow pytesseract -q
!apt-get install tesseract-ocr -q

# ================================
# IMPORTS
# ================================

import fitz
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pytesseract
import os
from google.colab import drive

drive.mount("/content/drive")

# ================================
# CONFIG
# ================================

PDF_DIR   = "/content/drive/MyDrive/glaze/swatches/1_2_pdfs"       # <-- folder containing all your PDFs
SAVE_BASE = "/content/drive/MyDrive/glaze/swatches/type_1_2_new"

VARIANCE_THRESHOLD = 21
MIN_SIZE = 1090
MAX_SIZE = 1200

GRID_ROWS = 4
GRID_COLS = 3

SWATCH_TOP    = 0.01
SWATCH_BOTTOM = 0.48
SWATCH_LEFT   = 0.54
SWATCH_RIGHT  = 1.0

# Bottom text region
TEXT_TOP    = 0.47
TEXT_BOTTOM = 0.65
TEXT_LEFT   = 0.65
TEXT_RIGHT  = 0.95

INSET = 8

SWATCH_MIN_SIZE = 110
SWATCH_MAX_SIZE = 125

# ================================
# HELPER FUNCTIONS
# ================================

def colour_variance(crop):
    return float(np.std(crop.astype(float)))


def overlap(a, b, threshold=0.5):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ix = max(0, min(ax+aw, bx+bw) - max(ax, bx))
    iy = max(0, min(ay+ah, by+bh) - max(ay, by))
    inter = ix * iy
    smaller = min(aw * ah, bw * bh)
    return inter / smaller > threshold if smaller > 0 else False


def detect_swatches(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    accepted = []
    rejected = []

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w < 20 or h < 20:
            continue

        crop = img_array[y:y+h, x:x+w]
        variance = colour_variance(crop)

        reasons = []
        if not (MIN_SIZE <= w <= MAX_SIZE and MIN_SIZE <= h <= MAX_SIZE):
            reasons.append(f"size {w}x{h}")
        if not (80 >= abs(w - h) >= 70):
            reasons.append(f"not square (diff={abs(w-h)})")
        if variance <= VARIANCE_THRESHOLD:
            reasons.append(f"flat var={variance:.0f}")

        if reasons:
            rejected.append((x, y, w, h, variance, ", ".join(reasons), crop))
        else:
            accepted.append((x, y, w, h, variance))

    filtered = []
    for c in sorted(accepted, key=lambda c: c[2]*c[3], reverse=True):
        if not any(overlap(c[:4], kept[:4]) for kept in filtered):
            filtered.append(c)

    return sorted(filtered, key=lambda c: (c[1], c[0])), rejected


def detect_swatches_in_cell(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    accepted = []
    rejected = []

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w < 20 or h < 20:
            continue

        crop = img_array[y:y+h, x:x+w]
        variance = colour_variance(crop)

        reasons = []
        if not (SWATCH_MIN_SIZE <= w <= SWATCH_MAX_SIZE and SWATCH_MIN_SIZE <= h <= SWATCH_MAX_SIZE):
            reasons.append(f"size {w}x{h}")
        if abs(w - h) > 10:
            reasons.append(f"not square (diff={abs(w-h)})")
        if variance <= VARIANCE_THRESHOLD:
            reasons.append(f"flat var={variance:.0f}")

        if reasons:
            rejected.append((x, y, w, h, variance, ", ".join(reasons), crop))
        else:
            accepted.append((x, y, w, h, variance))

    filtered = []
    for c in sorted(accepted, key=lambda c: c[2]*c[3], reverse=True):
        if not any(overlap(c[:4], kept[:4]) for kept in filtered):
            filtered.append(c)

    return sorted(filtered, key=lambda c: (c[1], c[0])), rejected


def clean_ocr(ocr_text, fallback):
    name = ocr_text.strip()
    name = name.replace("\n", "_")
    name = name.replace("-", "")
    name = name.replace(" ", "_")
    name = "".join(c for c in name if c.isalnum() or c in "_")
    return name if name else fallback


def unique_filename(save_dir, base_name):
    filename = f"{base_name}.png"
    counter = 1
    while os.path.exists(os.path.join(save_dir, filename)):
        filename = f"{base_name}_{counter}.png"
        counter += 1
    return filename

# ================================
# MAIN — PROCESS ALL PDFs
# ================================

pdf_files = [
    f for f in sorted(os.listdir(PDF_DIR))
    if f.lower().endswith(".pdf")
]

CONE_MAP = {
    1: "cone_6",
    2: "cone_10",
}

print(f"Found {len(pdf_files)} PDFs to process\n")

for pdf_name in pdf_files:
    pdf_path = os.path.join(PDF_DIR, pdf_name)
    original_filename = os.path.splitext(pdf_name)[0]

    print(f"\n{'='*60}")
    print(f"Processing: {pdf_name}")
    print(f"{'='*60}")

    doc = fitz.open(pdf_path)

    # ---- STAGE 1 + 2: DETECT GRIDS AND SPLIT INTO CELLS ----
    cells_by_page = {}

    for page_number, page in enumerate(doc, start=1):
        pix = page.get_pixmap(dpi=150)
        img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)

        accepted, _ = detect_swatches(img_array)
        print(f"  Page {page_number} — {len(accepted)} grid(s) found")

        cells_by_page[page_number] = []

        for grid_idx, (x, y, w, h, variance) in enumerate(accepted):
            grid_crop = img_array[y:y+h, x:x+w]
            gh, gw = grid_crop.shape[:2]

            cell_h = gh // GRID_ROWS
            cell_w = gw // GRID_COLS

            for row in range(GRID_ROWS):
                for col in range(GRID_COLS):
                    y1, y2 = row * cell_h, (row + 1) * cell_h
                    x1, x2 = col * cell_w, (col + 1) * cell_w
                    cell = grid_crop[y1:y2, x1:x2]
                    cell_num = row * GRID_COLS + col + 1
                    cells_by_page[page_number].append((grid_idx, cell_num, cell))

        print(f"    → {len(cells_by_page[page_number])} cells")

    # ---- STAGE 3: DETECT SWATCHES + OCR + SAVE ----
    for page_number, cells in cells_by_page.items():
        cone_folder = CONE_MAP.get(page_number, f"page_{page_number}")
        SAVE_DIR = os.path.join(SAVE_BASE, original_filename, cone_folder)
        os.makedirs(SAVE_DIR, exist_ok=True)

        print(f"\n  Page {page_number} → {cone_folder} ({len(cells)} cells)")

        total_swatches = 0
        total_saved    = 0

        for grid_idx, cell_num, cell in cells:
            cell_h, cell_w = cell.shape[:2]

            sy1 = int(cell_h * SWATCH_TOP)
            sy2 = int(cell_h * SWATCH_BOTTOM)
            sx1 = int(cell_w * SWATCH_LEFT)
            sx2 = int(cell_w * SWATCH_RIGHT)
            corner = cell[sy1:sy2, sx1:sx2]

            ty1 = int(cell_h * TEXT_TOP)
            ty2 = int(cell_h * TEXT_BOTTOM)
            tx1 = int(cell_w * TEXT_LEFT)
            tx2 = int(cell_w * TEXT_RIGHT)
            text_region = cell[ty1:ty2, tx1:tx2]

            ocr_text = pytesseract.image_to_string(
                Image.fromarray(text_region),
                config="--psm 6"
            ).strip()

            swatches, _ = detect_swatches_in_cell(corner)

            for idx, (x, y, w, h, variance) in enumerate(swatches):
                total_swatches += 1

                swatch = corner[y+INSET:y+h-INSET, x+INSET:x+w-INSET]

                fallback  = f"page{page_number:02d}_grid{grid_idx}_cell{cell_num:02d}_unknown"
                base_name = clean_ocr(ocr_text, fallback)
                filename  = unique_filename(SAVE_DIR, base_name)

                Image.fromarray(swatch).save(os.path.join(SAVE_DIR, filename))
                total_saved += 1
                print(f"      Saved: {filename}")

        print(f"    Swatches detected : {total_swatches}")
        print(f"    Files saved       : {total_saved}")
        print(f"    Files in folder   : {len(os.listdir(SAVE_DIR))}")

    doc.close()

print(f"\n{'='*60}")
print("All PDFs processed.")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
